In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense
from sklearn.metrics import classification_report

# 1. Create Sample Data for Training
texts = [
    "I absolutely love this product, it is amazing!",
    "This is the worst experience I have ever had.",
    "Fantastic service and great quality.",
    "Terrible, it broke after one day of use.",
    "Highly recommended, works perfectly.",
    "Do not buy this, it is a complete waste of money.",
    "I am very happy with my purchase.",
    "Awful customer support and bad product.",
    "Simply the best, exceeding all expectations.",
    "Disappointing and frustrating to use."
]
# Labels: 1 = Positive, 0 = Negative
labels = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

# 2. Preprocess the Text
vocab_size = 50
max_length = 10
embedding_dim = 16

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# 3. Build the BiLSTM Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    Bidirectional(LSTM(16)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid') # Binary classification
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 4. Train the Model
print("Training Model...")
model.fit(padded_sequences, labels, epochs=30, verbose=0) # verbose=0 hides epoch logs

# 5. Predictions and Simple Output
test_sentences = ["I really love it", "This is awful and bad"]
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_seq, maxlen=max_length, padding='post')

predictions = model.predict(test_padded, verbose=0)
print("\n--- Simple Output ---")
for i, sentence in enumerate(test_sentences):
    sentiment = "Positive" if predictions[i][0] > 0.5 else "Negative"
    print(f"Text: '{sentence}' | Predicted Sentiment: {sentiment} ({predictions[i][0]:.4f})")

# 6. Precision / Recall Matrix (Classification Report)
# Generating predictions on the training set to demonstrate the matrix
train_preds = (model.predict(padded_sequences, verbose=0) > 0.5).astype("int32")

print("\n--- Precision / Recall Matrix ---")
print(classification_report(labels, train_preds, target_names=['Negative', 'Positive']))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training Model...

--- Simple Output ---
Text: 'I really love it' | Predicted Sentiment: Positive (0.5667)
Text: 'This is awful and bad' | Predicted Sentiment: Positive (0.5403)

--- Precision / Recall Matrix ---
              precision    recall  f1-score   support

    Negative       1.00      0.60      0.75         5
    Positive       0.71      1.00      0.83         5

    accuracy                           0.80        10
   macro avg       0.86      0.80      0.79        10
weighted avg       0.86      0.80      0.79        10

